# 02 · Forecast backtest

Delivery day D is forecast at 12:00 on D-1. Test period: 2025-11-01 to 2026-10-31, monthly refit on an expanding window.

In [ ]:
from datetime import date

import polars as pl
import plotly.io as pio

from energy_forecast import features, forecast, plots
from energy_forecast.data import ROOT, load

pio.templates.default = "plotly_white"
df = features.build(load())

## Baselines

In [ ]:
naive = pl.concat([forecast.seasonal_naive(df, 24), forecast.seasonal_naive(df, 168)])
forecast.score(df, naive)

## LightGBM

In [ ]:
%%time
lgbm = forecast.lightgbm_backtest(df)
preds = pl.concat([naive, lgbm])
forecast.score(df, preds)

In [ ]:
plots.mae_by(df, preds, "month")

In [ ]:
plots.mae_by(df, preds, "hour")

In [ ]:
plots.forecast_lines(df, preds, date(2026, 6, 15), date(2026, 6, 21))

In [ ]:
plots.forecast_lines(df, preds, date(2026, 1, 12), date(2026, 1, 18))

## Save predictions

In [ ]:
out = ROOT / "data/processed"
out.mkdir(parents=True, exist_ok=True)
preds.write_parquet(out / "backtest.parquet")